In [88]:
print("ok")

ok


In [89]:
from langchain.agents import create_agent
from langchain.agents import AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.tools import tool, ToolRuntime
from sibyl_prompt import SIBYL_PROMPT



In [90]:
from dotenv import load_dotenv
load_dotenv()
import os
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

In [91]:

from langchain_anthropic import ChatAnthropic 
model = ChatAnthropic(
    model="claude-haiku-4-5",
    temperature=0,
    )

In [92]:
client1 = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "/Users/oluwaferanmioyelude/Documents/Coding Projects/Sibyl/.venv/bin/python",
            "args": [
                "/Users/oluwaferanmioyelude/Documents/Coding Projects/Sibyl/mcp_server.py"
            ],
        }
    }
)

In [93]:
mcp_tools = await client1.get_tools()

In [94]:
from dataclasses import dataclass
@dataclass
class CustomState(AgentState):
    username: str
    academic_standing: str
    university: str

In [95]:

from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_username(username: str, runtime: ToolRuntime) -> Command:
    """ Update the username of the user in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "username": username,
        "messages": [ToolMessage("Successfully updated user's username", tool_call_id=runtime.tool_call_id)]
    })
@tool
def update_user_university(university: str, runtime: ToolRuntime) -> Command:
    """ Update the university of the user in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "university": university,
        "messages": [ToolMessage("Successfully updated user's university", tool_call_id=runtime.tool_call_id)]
    })
@tool
def update_user_academic_standing(academic_standing : str, runtime: ToolRuntime) -> Command:
    """ Update the academic standing of the user(like freshman, sophmore) in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "academic_standing": academic_standing,
        "messages": [ToolMessage("Successfully updated user's academic standing", tool_call_id=runtime.tool_call_id)]
    })

In [96]:
agent= create_agent(
    model=model, 
    tools=[update_user_academic_standing, update_user_university,update_username, *mcp_tools],
    system_prompt=SIBYL_PROMPT,
    checkpointer=InMemorySaver(),
    state_schema= CustomState
    )

In [97]:
config={"configurable": {"thread_id":"1"}}
from langchain.messages import HumanMessage
question=HumanMessage(content="Hello. I am Oluwaferanmi Oyelude.")
response1=await agent.ainvoke({"messages": [question]}, config)

In [98]:
from pprint import pprint
pprint(response1)

{'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='97d5a6d3-bb80-4c1c-9a50-9f098093c2c0'),
              AIMessage(content="Hey Oluwaferanmi! 🎯 I'm **SIBYL**, your Academic Strategist Agent.\n\nNice to meet you. I'm here to help you dominate your semester—scanning syllabi, strategizing deadlines, and optimizing your calendar so you crush your GPA without burning out.\n\nLet me get some quick intel on you:\n\n1. **What's your username?** (for tracking purposes)\n2. **Which university are you at?**\n3. **What's your current academic standing?** (Freshman, Sophomore, Junior, Senior, etc.)\n\nOnce I have that locked in, we can move forward. Got any syllabi ready to scan, or are we setting up your strategy first?", additional_kwargs={}, response_metadata={'id': 'msg_01PjCNRLpxfHoGBBuioNs8QR', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_i

In [99]:
question=HumanMessage(content="My name is Oluwaferanmi Oyelude, my university is Howard University and i am a freshman")
response2=await agent.ainvoke({"messages": [question]}, config)

In [100]:
from pprint import pprint
pprint(response2)

{'academic_standing': 'Freshman',
 'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='97d5a6d3-bb80-4c1c-9a50-9f098093c2c0'),
              AIMessage(content="Hey Oluwaferanmi! 🎯 I'm **SIBYL**, your Academic Strategist Agent.\n\nNice to meet you. I'm here to help you dominate your semester—scanning syllabi, strategizing deadlines, and optimizing your calendar so you crush your GPA without burning out.\n\nLet me get some quick intel on you:\n\n1. **What's your username?** (for tracking purposes)\n2. **Which university are you at?**\n3. **What's your current academic standing?** (Freshman, Sophomore, Junior, Senior, etc.)\n\nOnce I have that locked in, we can move forward. Got any syllabi ready to scan, or are we setting up your strategy first?", additional_kwargs={}, response_metadata={'id': 'msg_01PjCNRLpxfHoGBBuioNs8QR', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {

In [101]:
question=HumanMessage(content="I want you to scan my syllabi. This is the path to it: /Users/oluwaferanmioyelude/Documents/Syllabus")
response3=await agent.ainvoke({"messages": [question]}, config)

In [102]:
pprint(response3)

{'academic_standing': 'Freshman',
 'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='97d5a6d3-bb80-4c1c-9a50-9f098093c2c0'),
              AIMessage(content="Hey Oluwaferanmi! 🎯 I'm **SIBYL**, your Academic Strategist Agent.\n\nNice to meet you. I'm here to help you dominate your semester—scanning syllabi, strategizing deadlines, and optimizing your calendar so you crush your GPA without burning out.\n\nLet me get some quick intel on you:\n\n1. **What's your username?** (for tracking purposes)\n2. **Which university are you at?**\n3. **What's your current academic standing?** (Freshman, Sophomore, Junior, Senior, etc.)\n\nOnce I have that locked in, we can move forward. Got any syllabi ready to scan, or are we setting up your strategy first?", additional_kwargs={}, response_metadata={'id': 'msg_01PjCNRLpxfHoGBBuioNs8QR', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {

In [103]:
pprint(response3["messages"][-2].content)

[{'id': 'lc_cd26ba28-0095-4878-bd3e-3dfdb829f90f',
  'text': '{\n'
          '  "courses": [\n'
          '    {\n'
          '      "course_name": "ENGW 104 - Writing, Literacy, and '
          'Discourse",\n'
          '      "events": [\n'
          '        {\n'
          '          "title": "Personal Declaration Essay",\n'
          '          "type": "assignment",\n'
          '          "due_date": "2026-01-21",\n'
          '          "priority": "medium",\n'
          '          "strategy": {\n'
          '            "action": "Start Working",\n'
          '            "start_date": "2026-01-19",\n'
          '            "reasoning": "2 days lead time for assignments"\n'
          '          }\n'
          '        },\n'
          '        {\n'
          '          "title": "Personal Declaration Essay: Second Draft",\n'
          '          "type": "assignment",\n'
          '          "due_date": "2026-02-06",\n'
          '          "priority": "medium",\n'
          '    

In [104]:
question=HumanMessage(content="Can you list the events you found?")
response4=await agent.ainvoke({"messages": [question]}, config)

In [105]:
pprint(response4)

{'academic_standing': 'Freshman',
 'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='97d5a6d3-bb80-4c1c-9a50-9f098093c2c0'),
              AIMessage(content="Hey Oluwaferanmi! 🎯 I'm **SIBYL**, your Academic Strategist Agent.\n\nNice to meet you. I'm here to help you dominate your semester—scanning syllabi, strategizing deadlines, and optimizing your calendar so you crush your GPA without burning out.\n\nLet me get some quick intel on you:\n\n1. **What's your username?** (for tracking purposes)\n2. **Which university are you at?**\n3. **What's your current academic standing?** (Freshman, Sophomore, Junior, Senior, etc.)\n\nOnce I have that locked in, we can move forward. Got any syllabi ready to scan, or are we setting up your strategy first?", additional_kwargs={}, response_metadata={'id': 'msg_01PjCNRLpxfHoGBBuioNs8QR', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {

In [106]:
question=HumanMessage(content="Can you load all the high priority events to my calendar?")
response5=await agent.ainvoke({"messages": [question]}, config)

In [107]:
pprint(response5["messages"][-2].content)

[{'id': 'lc_c28b7a37-55f9-47be-8bc7-4617472d21db',
  'text': 'Success: Event created! Link: '
          'https://www.google.com/calendar/event?eid=dWMya3EzN2Vhc29oZTg5bTUxdmRrYjdpaGsgb2x1d2FmZXJhbm1pb3llbHVkZTIwMjRAbQ\n'
          'Success: Event created! Link: '
          'https://www.google.com/calendar/event?eid=OXYwOWYycjNhdGVzMjV0OGg0NmtzNGN0ZW8gb2x1d2FmZXJhbm1pb3llbHVkZTIwMjRAbQ\n'
          'Success: Event created! Link: '
          'https://www.google.com/calendar/event?eid=bG9lZ2JvNjFtcmJnZnFxMTBiNGVicGxrOTAgb2x1d2FmZXJhbm1pb3llbHVkZTIwMjRAbQ',
  'type': 'text'}]


In [108]:
question=HumanMessage(content="Load the rest as well")
response6=await agent.ainvoke({"messages": [question]}, config)

In [110]:
pprint(response6)

{'academic_standing': 'Freshman',
 'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='97d5a6d3-bb80-4c1c-9a50-9f098093c2c0'),
              AIMessage(content="Hey Oluwaferanmi! 🎯 I'm **SIBYL**, your Academic Strategist Agent.\n\nNice to meet you. I'm here to help you dominate your semester—scanning syllabi, strategizing deadlines, and optimizing your calendar so you crush your GPA without burning out.\n\nLet me get some quick intel on you:\n\n1. **What's your username?** (for tracking purposes)\n2. **Which university are you at?**\n3. **What's your current academic standing?** (Freshman, Sophomore, Junior, Senior, etc.)\n\nOnce I have that locked in, we can move forward. Got any syllabi ready to scan, or are we setting up your strategy first?", additional_kwargs={}, response_metadata={'id': 'msg_01PjCNRLpxfHoGBBuioNs8QR', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {